
# classification_upgrade.ipynb

Апгрейд пайплайна pair-classification для SCE-Net.

Этот ноутбук:
1. Интерпретирует полученные логи обучения.
2. Добавляет аккуратные улучшения (loss, scheduler, threshold tuning, early stopping).
3. Запускает **несколько экспериментов в цикле** и сохраняет отдельные чекпоинты.
4. Упрощает сравнение моделей на val/test в единой таблице.

---

## Предпосылки
Предполагается, что уже есть:
- `train_triplets`, `val_triplets`
- определения классов `SCEPairClassifier`, `PairClsConfig`
- функции `build_image_cache`, `compute_binary_metrics`
- `processor`, `device`

Если нет — сначала выполните базовый ноутбук `sce_net_fashion_compatibility_pair_classification.ipynb`.



## 0) Интерпретация ваших логов (короткий вывод)

По вашим метрикам за 15 эпох:
- **val ROC-AUC вырос ~0.822 -> ~0.921**,
- **val PR-AUC вырос ~0.552 -> ~0.769**,
- **val logloss стабильно падал ~0.342 -> ~0.242**,
- `train` и `val` идут близко (без явного сильного overfit).

Это очень хороший признак: даже при `freeze_backbone=True` модель научилась лучше ранжировать и калибровать пары.

Почему получилось лучше триплетов (`~0.8275 ROC-AUC`):
- задача формулируется напрямую как бинарная совместимость,
- BCE даёт стабильный сигнал на каждом примере,
- порог и вероятность интерпретируемы напрямую.

Что улучшать дальше:
1. `pos_weight` в BCE (если дисбаланс классов).
2. LR scheduler (cosine + warmup).
3. Чуть длиннее обучение с early stopping.
4. Частичный/unfreeze backbone в конце (2-stage fine-tuning).
5. Сохранение нескольких лучших моделей и сравнение ансамблей.


In [ ]:

import os
import copy
import math
import random
from dataclasses import dataclass, asdict
from typing import Dict, List

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import f1_score, accuracy_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

os.makedirs('checkpoints_classification_upgrade', exist_ok=True)


In [ ]:

# Безопасная проверка контекста
required = ['train_triplets', 'val_triplets', 'SCEPairClassifier', 'build_image_cache', 'compute_binary_metrics', 'processor', 'device']
for v in required:
    if v not in globals():
        raise RuntimeError(f'Missing required object: {v}. Сначала запустите базовый ноутбук.')

print('All required objects found.')


In [ ]:

def triplets_to_pair_df(triplets_df: pd.DataFrame) -> pd.DataFrame:
    pos_df = pd.DataFrame({
        'sku1': triplets_df['anchor_sku'].astype(str),
        'sku2': triplets_df['pos_sku'].astype(str),
        'sku1_path': triplets_df['anchor_path'].astype(str),
        'sku2_path': triplets_df['pos_path'].astype(str),
        'y': 1,
    })
    neg_df = pd.DataFrame({
        'sku1': triplets_df['anchor_sku'].astype(str),
        'sku2': triplets_df['neg_sku'].astype(str),
        'sku1_path': triplets_df['anchor_path'].astype(str),
        'sku2_path': triplets_df['neg_path'].astype(str),
        'y': 0,
    })
    pair_df = pd.concat([pos_df, neg_df], axis=0, ignore_index=True)
    pair_df = pair_df.drop_duplicates(subset=['sku1','sku2','sku1_path','sku2_path','y']).reset_index(drop=True)
    pair_df = pair_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    return pair_df

train_pairs = triplets_to_pair_df(train_triplets)
val_pairs = triplets_to_pair_df(val_triplets)

print('train_pairs:', train_pairs.shape, 'pos_rate=', train_pairs['y'].mean())
print('val_pairs:', val_pairs.shape, 'pos_rate=', val_pairs['y'].mean())


In [ ]:

class PairImageDataset(Dataset):
    def __init__(self, pair_df: pd.DataFrame, image_cache: Dict[str, np.ndarray]):
        self.df = pair_df.reset_index(drop=True)
        self.image_cache = image_cache

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        p1, p2 = str(row['sku1_path']), str(row['sku2_path'])
        if p1 not in self.image_cache or p2 not in self.image_cache:
            raise KeyError(f'path missing in cache: {p1}, {p2}')
        return self.image_cache[p1], self.image_cache[p2], np.float32(row['y'])


def make_pair_collate(processor):
    def collate_fn(batch):
        imgs1, imgs2, ys = zip(*batch)
        px1 = processor(images=list(imgs1), return_tensors='pt')['pixel_values']
        px2 = processor(images=list(imgs2), return_tensors='pt')['pixel_values']
        ys = torch.tensor(ys, dtype=torch.float32)
        return px1, px2, ys
    return collate_fn


In [ ]:

# Кешируем train+val изображения
all_paths = list(set(pd.concat([
    train_pairs['sku1_path'], train_pairs['sku2_path'],
    val_pairs['sku1_path'], val_pairs['sku2_path'],
], axis=0).astype(str).tolist()))

IMAGE_CACHE_UPG = build_image_cache(all_paths, target_size=224, verbose=True, max_workers=16)

train_ds = PairImageDataset(train_pairs, IMAGE_CACHE_UPG)
val_ds = PairImageDataset(val_pairs, IMAGE_CACHE_UPG)

collate_fn = make_pair_collate(processor)


In [ ]:

@dataclass
class UpgradeConfig:
    name: str
    base_cfg: dict
    lr: float = 1e-4
    weight_decay: float = 1e-4
    epochs: int = 20
    batch_size: int = 256
    freeze_backbone: bool = True
    lambda_l1: float = 1e-4
    lambda_l2: float = 1e-4
    use_pos_weight: bool = True
    label_smoothing: float = 0.0
    warmup_ratio: float = 0.1
    early_stop_patience: int = 4


def get_pos_weight(pair_df: pd.DataFrame) -> float:
    pos = float((pair_df['y'] == 1).sum())
    neg = float((pair_df['y'] == 0).sum())
    if pos == 0:
        return 1.0
    return neg / pos


class SmoothBCE(torch.nn.Module):
    def __init__(self, smoothing: float = 0.0, pos_weight: torch.Tensor = None):
        super().__init__()
        self.smoothing = float(smoothing)
        self.pos_weight = pos_weight

    def forward(self, logits: torch.Tensor, targets: torch.Tensor):
        if self.smoothing > 0:
            targets = targets * (1.0 - self.smoothing) + 0.5 * self.smoothing
        return F.binary_cross_entropy_with_logits(logits, targets, pos_weight=self.pos_weight)


In [ ]:

def create_loaders(batch_size: int, num_workers: int = 16, prefetch_factor: int = 2, persistent_workers: bool = True):
    loader_kwargs = dict(
        batch_size=batch_size,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        collate_fn=collate_fn,
    )
    if num_workers > 0:
        loader_kwargs['prefetch_factor'] = prefetch_factor
        loader_kwargs['persistent_workers'] = persistent_workers

    train_loader = DataLoader(train_ds, shuffle=True, **loader_kwargs)
    val_loader = DataLoader(val_ds, shuffle=False, **loader_kwargs)
    return train_loader, val_loader


@torch.no_grad()
def predict_proba(model, loader):
    model.eval()
    probs, ys = [], []
    for px1, px2, y in tqdm(loader, leave=False):
        px1 = px1.to(device, non_blocking=True)
        px2 = px2.to(device, non_blocking=True)
        logits, _, _ = model(px1, px2)
        probs.append(torch.sigmoid(logits).cpu().numpy())
        ys.append(y.numpy())
    return np.concatenate(probs), np.concatenate(ys).astype(int)


def best_threshold_by_f1(y_true: np.ndarray, y_prob: np.ndarray):
    thresholds = np.linspace(0.1, 0.9, 81)
    best_thr, best_f1 = 0.5, -1
    for thr in thresholds:
        pred = (y_prob >= thr).astype(int)
        f1 = f1_score(y_true, pred)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = float(thr)
    return best_thr, best_f1


In [ ]:

def run_epoch_upgrade(model, loader, criterion, optimizer=None, scaler=None, lambda_l1=1e-4, lambda_l2=1e-4):
    is_train = optimizer is not None
    model.train(is_train)

    losses = []
    y_prob_all, y_true_all = [], []

    for px1, px2, y in tqdm(loader, leave=False):
        px1 = px1.to(device, non_blocking=True)
        px2 = px2.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        with torch.set_grad_enabled(is_train):
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                logits, e1, e2 = model(px1, px2)
                bce = criterion(logits, y)
                l1 = model.encoder.condition_masks.abs().mean()
                l2 = (e1.pow(2).mean() + e2.pow(2).mean()) / 2.0
                loss = bce + lambda_l1 * l1 + lambda_l2 * l2

            if is_train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        losses.append(loss.detach().item())
        y_prob_all.append(torch.sigmoid(logits).detach().cpu().numpy())
        y_true_all.append(y.detach().cpu().numpy())

    y_prob = np.concatenate(y_prob_all)
    y_true = np.concatenate(y_true_all).astype(int)

    m = compute_binary_metrics(y_true, y_prob, thr=0.5)
    m['loss'] = float(np.mean(losses))
    return m


In [ ]:

def train_one_experiment(exp: UpgradeConfig):
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    cfg_local = copy.deepcopy(exp.base_cfg)
    cfg_local['freeze_backbone'] = exp.freeze_backbone

    model = SCEPairClassifier(type('obj', (object,), cfg_local)()).to(device)

    if exp.freeze_backbone:
        for p in model.encoder.clip.parameters():
            p.requires_grad = False

    train_loader, val_loader = create_loaders(batch_size=exp.batch_size)

    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=exp.lr, weight_decay=exp.weight_decay)
    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

    steps_per_epoch = len(train_loader)
    total_steps = max(1, steps_per_epoch * exp.epochs)
    warmup_steps = int(total_steps * exp.warmup_ratio)

    def lr_lambda(step):
        if step < warmup_steps:
            return float(step + 1) / float(max(1, warmup_steps))
        progress = (step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    pos_weight = None
    if exp.use_pos_weight:
        pos_weight = torch.tensor([get_pos_weight(train_pairs)], device=device, dtype=torch.float32)
    criterion = SmoothBCE(smoothing=exp.label_smoothing, pos_weight=pos_weight)

    best_score = -np.inf
    best_state = None
    bad_epochs = 0
    history = []

    ckpt_path = f"checkpoints_classification_upgrade/{exp.name}.pt"

    global_step = 0
    for epoch in range(1, exp.epochs + 1):
        tr = run_epoch_upgrade(model, train_loader, criterion, optimizer=optimizer, scaler=scaler, lambda_l1=exp.lambda_l1, lambda_l2=exp.lambda_l2)
        va = run_epoch_upgrade(model, val_loader, criterion, optimizer=None, scaler=scaler, lambda_l1=exp.lambda_l1, lambda_l2=exp.lambda_l2)

        # step scheduler per batch-equivalent count (approx)
        for _ in range(steps_per_epoch):
            scheduler.step()
            global_step += 1

        row = {
            'epoch': epoch,
            'train_loss': tr['loss'],
            'val_loss': va['loss'],
            'train_roc_auc': tr['roc_auc'],
            'val_roc_auc': va['roc_auc'],
            'train_pr_auc': tr['pr_auc'],
            'val_pr_auc': va['pr_auc'],
            'val_f1@0.5': va['f1@0.5'],
            'val_acc@0.5': va['acc@0.5'],
            'val_logloss': va['logloss'],
            'lr': optimizer.param_groups[0]['lr'],
        }
        history.append(row)

        score = va['pr_auc']
        if score > best_score:
            best_score = score
            bad_epochs = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            torch.save({
                'model_state': model.state_dict(),
                'config': asdict(exp),
                'best_val_pr_auc': best_score,
            }, ckpt_path)
            print(f"[{exp.name}] epoch={epoch} -> saved best: {ckpt_path}")
        else:
            bad_epochs += 1

        print(
            f"[{exp.name}] [Epoch {epoch:02d}] "
            f"train pr={tr['pr_auc']:.4f} auc={tr['roc_auc']:.4f} | "
            f"val pr={va['pr_auc']:.4f} auc={va['roc_auc']:.4f} f1={va['f1@0.5']:.4f}"
        )

        if bad_epochs >= exp.early_stop_patience:
            print(f"[{exp.name}] early stopping at epoch={epoch}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    val_prob, val_true = predict_proba(model, val_loader)
    best_thr, best_f1 = best_threshold_by_f1(val_true, val_prob)
    val_metrics = compute_binary_metrics(val_true, val_prob, thr=best_thr)

    return {
        'name': exp.name,
        'history': pd.DataFrame(history),
        'best_threshold': best_thr,
        'best_f1_val': best_f1,
        'val_metrics_at_best_thr': val_metrics,
        'checkpoint_path': ckpt_path,
    }


In [ ]:

# Базовый конфиг (как у вас)
base_cfg = {
    'hf_model_name': 'patrickjohncyh/fashion-clip',
    'num_conditions': 5,
    'condition_hidden': 1024,
    'dropout': 0.1,
    'pair_hidden': 1024,
    'batch_size': 256,
    'lr': 1e-4,
    'weight_decay': 1e-4,
    'epochs': 15,
    'use_amp': True,
    'freeze_backbone': True,
    'lambda_l1': 1e-4,
    'lambda_l2': 1e-4,
    'num_workers': 16,
    'prefetch_factor': 2,
    'persistent_workers': True,
}

experiments = [
    UpgradeConfig(name='exp_baseline_freeze', base_cfg=base_cfg, freeze_backbone=True, lr=1e-4, epochs=15, use_pos_weight=False, label_smoothing=0.0),
    UpgradeConfig(name='exp_freeze_posweight', base_cfg=base_cfg, freeze_backbone=True, lr=1e-4, epochs=18, use_pos_weight=True, label_smoothing=0.0),
    UpgradeConfig(name='exp_freeze_posweight_smooth', base_cfg=base_cfg, freeze_backbone=True, lr=8e-5, epochs=20, use_pos_weight=True, label_smoothing=0.02),
    # Осторожный partial fine-tuning (если хватит GPU времени)
    UpgradeConfig(name='exp_unfreeze_low_lr', base_cfg=base_cfg, freeze_backbone=False, lr=2e-5, epochs=8, use_pos_weight=True, label_smoothing=0.0),
]

experiments


In [ ]:

results = []
for exp in experiments:
    out = train_one_experiment(exp)
    results.append(out)

print('Finished experiments:', [r['name'] for r in results])


In [ ]:

leaderboard_rows = []
for r in results:
    m = r['val_metrics_at_best_thr']
    leaderboard_rows.append({
        'name': r['name'],
        'checkpoint_path': r['checkpoint_path'],
        'best_threshold': r['best_threshold'],
        'val_roc_auc': m['roc_auc'],
        'val_pr_auc': m['pr_auc'],
        'val_f1': m['f1@0.5'],
        'val_acc': m['acc@0.5'],
        'val_logloss': m['logloss'],
    })

leaderboard = pd.DataFrame(leaderboard_rows).sort_values(['val_pr_auc','val_roc_auc'], ascending=False).reset_index(drop=True)
leaderboard


In [ ]:

# Сравнение на test_final_v2.csv (если доступен)
if os.path.exists('test_final_v2.csv'):
    test = pd.read_csv('test_final_v2.csv')
    print('test shape:', test.shape)

    all_paths_test = list(set(pd.concat([test['sku1_path'], test['sku2_path']], axis=0).astype(str).tolist()))
    IMAGE_CACHE_TEST = build_image_cache(all_paths_test, target_size=224, verbose=True, max_workers=16)

    def infer_with_checkpoint(ckpt_path: str, threshold: float):
        model = SCEPairClassifier(type('obj', (object,), base_cfg)()).to(device)
        ckpt = torch.load(ckpt_path, map_location=device)
        state = ckpt['model_state'] if isinstance(ckpt, dict) and 'model_state' in ckpt else ckpt
        model.load_state_dict(state)
        model.eval()

        probs = []
        rows = test[['sku1_path','sku2_path']].astype(str).values.tolist()
        for i in tqdm(range(0, len(rows), 128), leave=False, desc=f'infer:{os.path.basename(ckpt_path)}'):
            chunk = rows[i:i+128]
            imgs1, imgs2 = [], []
            for p1, p2 in chunk:
                imgs1.append(IMAGE_CACHE_TEST[p1])
                imgs2.append(IMAGE_CACHE_TEST[p2])
            px1 = processor(images=imgs1, return_tensors='pt')['pixel_values'].to(device)
            px2 = processor(images=imgs2, return_tensors='pt')['pixel_values'].to(device)
            logits, _, _ = model(px1, px2)
            probs.extend(torch.sigmoid(logits).cpu().numpy().tolist())

        probs = np.array(probs, dtype=np.float32)
        pred_y = (probs >= threshold).astype(int)
        return probs, pred_y

    test_rows = []
    for _, row in leaderboard.iterrows():
        probs, pred_y = infer_with_checkpoint(row['checkpoint_path'], row['best_threshold'])

        if 'target' in test.columns:
            y_true = test['target'].map({'Normal':1, 'Bad':0})
            valid = ~y_true.isna()
            y_true = y_true[valid].astype(int).values
            y_prob = probs[valid.values]
            m = compute_binary_metrics(y_true, y_prob, thr=row['best_threshold'])
            test_rows.append({'name': row['name'], 'test_roc_auc': m['roc_auc'], 'test_pr_auc': m['pr_auc'], 'test_f1': m['f1@0.5'], 'test_acc': m['acc@0.5'], 'test_logloss': m['logloss']})
        else:
            test_rows.append({'name': row['name'], 'test_roc_auc': np.nan, 'test_pr_auc': np.nan, 'test_f1': np.nan, 'test_acc': np.nan, 'test_logloss': np.nan})

        out_df = test.copy()
        out_df[f"pred_prob_{row['name']}"] = probs
        out_df[f"pred_target_{row['name']}"] = np.where(pred_y == 1, 'Normal', 'Bad')
        out_df.to_csv(f"test_predictions_{row['name']}.csv", index=False)

    test_leaderboard = pd.DataFrame(test_rows).sort_values(['test_pr_auc','test_roc_auc'], ascending=False)
    display(test_leaderboard)
else:
    print('test_final_v2.csv not found, skip test comparison block.')



## Рекомендованный план развития (практика)

1. Сначала сравнить `exp_baseline_freeze` vs `exp_freeze_posweight`.
2. Если gain стабильный — добавить `label_smoothing`.
3. Только потом запускать `unfreeze` с маленьким LR.
4. Финально взять top-2 модели по `val_pr_auc` и проверить на test.
5. При близких метриках выбирать модель с лучшим `logloss` (лучше калибровка вероятностей).
